# Phase 2: YOLOv8 Classifier Training

This notebook downloads the Voxel51/Safe_and_Unsafe_Behaviours dataset, extracts 3 frames from each video, and trains a lightweight YOLOv8 Nano classifier for the specific policy behaviors.

In [ ]:
%pip install ultralytics

In [ ]:
import os
import cv2
from pathlib import Path
from datasets import load_dataset
from ultralytics import YOLO
from IPython.display import display, Image

In [ ]:
DATA_DIR = Path("../datasets/classification")

def extract_multiple_frames(video_path: str, base_output_path: Path, num_frames=3):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames > 0:
        # Extract at 25%, 50%, and 75% of the video to avoid start/end fades
        fractions = [0.25, 0.50, 0.75]
        for i, frac in enumerate(fractions):
            frame_pos = int(total_frames * frac)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_pos)
            ret, frame = cap.read()
            if ret:
                # Append _f0, _f1, _f2 to the filename
                out_name = f"{base_output_path.stem}_f{i}{base_output_path.suffix}"
                final_out = base_output_path.parent / out_name
                cv2.imwrite(str(final_out), frame)
    cap.release()

def prep_dataset():
    print("Loading full dataset...")
    ds = load_dataset("Voxel51/Safe_and_Unsafe_Behaviours")
    
    for split in ['train', 'test']:
        out_split = 'train' if split == 'train' else 'val'
        for item in ds[split]:
            vid_path = item.get('video') or item.get('path')
            if not vid_path:
                continue
                
            filename = Path(vid_path).name
            class_id = filename.split('_')[0]
            
            class_dir = DATA_DIR / out_split / class_id
            class_dir.mkdir(parents=True, exist_ok=True)
            
            out_img_base = class_dir / f"{filename.split('.')[0]}.jpg"
            # Check if the first frame was already extracted to avoid repeating work
            check_img = class_dir / f"{filename.split('.')[0]}_f0.jpg"
            
            if not check_img.exists():
                extract_multiple_frames(vid_path, out_img_base, num_frames=3)
                
    print("Full dataset prepared for YOLO format with 3 frames per video.")

In [ ]:
if not DATA_DIR.exists():
    prep_dataset()

### 1. Train the Classifier

In [ ]:
model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=str(DATA_DIR.absolute()),
    epochs=5,
    imgsz=224,
    batch=16,
    project="../outputs",
    name="yolo_classifier",
    device=0
)

### 2. Evaluate the Model

In [ ]:
# Run validation on the test/val split
metrics = model.val()
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

### 3. Visualizing Performance

In [ ]:
# Display the automatically generated confusion matrix
cm_path = Path("../outputs/yolo_classifier/confusion_matrix.png")
if cm_path.exists():
    display(Image(filename=str(cm_path), width=600))
else:
    print("Confusion matrix not found. Check outputs directory.")